In [ ]:
import pandas as pd
import numpy as np
import os
os.environ["CUDA_VISIBLE_DEVICES"]="3"
import sys
#import torch
import collections
import nltk
import pickle
import re
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
import string
punct = string.punctuation
import sys
import os
from os import path

#sys.path.append('../pyTsetlinMachineParallel/') 
#sys.path.append('../pyTsetlinMachine/') 
#sys.path.append('../pyTsetlinMachine_clauses/')  
#sys.path.append('/data/annotated_pytsetlin_machine-master/pyTsetlinMachine_score/')  
#from PyTsetlinMachineCUDA.tm import MultiClassTsetlinMachine
from PyTsetlinMachineCUDA.tm import MultiClassTsetlinMachine
from sklearn.metrics import f1_score
from os.path import isfile, join
import string
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
#from tm import MultiClassTsetlinMachine
from os import listdir
from time import time
from sklearn import preprocessing
import nltk
nltk.download('stopwords')


path_train= 'train.txt'
path_test='test.txt'
'''
df= open(path, "r", encoding="utf-8")
text = df.readlines()
text_new=[]
label=[]
for line in text:
    text = line.split('\n')
    text_label= line.replace('"', '').replace('\n', '').split('\t')
    label.append(text_label[0]) 
    text_new.append(text_label[1])
#print((text_new))
'''

f = open(path_train, "r", encoding="utf-8")
lines = f.readlines()
f.close()

doc_name_list_train = []
doc_content_list_train = []
for line in lines:
    line = line.strip()
    label = line[:line.find('\t')]
    content = line[line.find('\t') + 1:]
    #string = str(doc_id) + '\t' + 'train' + '\t' + label
    doc_name_list_train.append(label)
    doc_content_list_train.append(content)

f = open(path_test, "r", encoding="utf-8")
lines = f.readlines()
f.close()
doc_name_list_test = []
doc_content_list_test = []
for line in lines:
    line = line.strip()
    label = line[:line.find('\t')]
    content = line[line.find('\t') + 1:]
    #string = str(doc_id) + '\t' + 'test' + '\t' + label
    doc_name_list_test.append(label)
    doc_content_list_test.append(content)


print(len(doc_name_list_train))
print(len(doc_name_list_test))

label_encoder = preprocessing.LabelEncoder()
label= label_encoder.fit_transform(doc_name_list_train+doc_name_list_test)
label_train = label[:5485]
label_test= label[5485:]

print(len(doc_content_list_train), len(label_train), len(doc_content_list_test), len(label_test))

import string
punct = string.punctuation
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
from nltk.corpus import stopwords
stopwords = set(stopwords.words('english'))

def preprocess(words):
    # we'll make use of python's translate function,that maps one set of characters to another
    # we create an empty mapping table, the third argument allows us to list all of the characters
    # to remove during the translation process

    # first we will try to filter out some  unnecessary data like tabs
    table = str.maketrans('', '', '\t')
    
    words = [word.translate(table) for word in words]
    words= [lemmatizer.lemmatize(word) for word in words]

    punctuations = (string.punctuation).replace("'", "")
    # the character: ' appears in a lot of stopwords and changes meaning of words if removed
    # hence it is removed from the list of symbols that are to be discarded from the documents
    trans_table = str.maketrans('', '', punctuations)
    stripped_words = [word.translate(trans_table) for word in words]

    # some white spaces may be added to the list of words, due to the translate function & nature of our documents
    # we remove them belowr
    words = [str for str in stripped_words if str]

    # some words are quoted in the documents & as we have not removed ' to maintain the integrity of some stopwords
    # we try to unquote such words below
    p_words = []
    for word in words:
        if (word[0] and word[len(word) - 1] == "'"):
            word = word[1:len(word) - 1]
        elif (word[0] == "'"):
            word = word[1:len(word)]
        else:
            word = word
        p_words.append(word)

    words = p_words.copy()

    # we will also remove just-numeric strings as they do not have any significant meaning in text classification
    words = [word for word in words if not word.isdigit()]

    # we will also remove single character strings
    words = [word for word in words if not len(word) == 1]

    # after removal of so many characters it may happen that some strings have become blank, we remove those
    words = [str for str in words if str]

    # we also normalize the cases of our words
    words = [word.lower() for word in words]

    # we try to remove words with only 2 characters
    words = [word for word in words if len(word) > 2]

    return words

def flatten(list):
    new_list = []
    for i in list:
        for j in i:
            new_list.append(j)
    return new_list

def remove_stopwords(words):
    words = [word for word in words if not word in stopwords]
    return words

def tokenize_line(line):
    text = line[0:len(line)-1].strip().split(" ")
    text= remove_stopwords(text)
    text= preprocess(text)
    return text

def tokenize(text):
    string = re.sub(r"[^A-Za-z0-9(),!?\'\`]", " ", text)
    string = re.sub(r"\'s", " \'s", string)
    string = re.sub(r"\'ve", " \'ve", string)
    string = re.sub(r"n\'t", " n\'t", string)
    string = re.sub(r"\'re", " \'re", string)
    string = re.sub(r"\'d", " \'d", string)
    string = re.sub(r"\'ll", " \'ll", string)
    string = re.sub(r",", " , ", string)
    string = re.sub(r"!", " ! ", string)
    string = re.sub(r"\(", " \( ", string)
    string = re.sub(r"\)", " \) ", string)
    string = re.sub(r"\?", " \? ", string)
    string = re.sub(r"\s{2,}", " ", string)
    text = tokenize_line(string)
    return text

train_total_list= [tokenize(t) for t in doc_content_list_train]
train_label= np.array(label_train)

test_total_list= [tokenize(t) for t in doc_content_list_test]
test_label= np.array(label_test)

print("test_label", test_label)
np_total=  train_total_list + test_total_list
words, counts = np.unique(flatten(np_total), return_counts=True)
freq, wrds = (list(i) for i in zip(*(sorted(zip(counts, words), reverse=True))))

n = 10000
features = wrds[0:n]

if not(os.path.exists('./r8.pkl')):
  word_set=set(wrds)
  word_idx = dict((c, i + 1) for i, c in enumerate(word_set))
  np_word_idx= np.asarray(word_idx)
  reverse_word_map = dict(map(reversed, word_idx.items()))
  
  output= open("./r8.pkl", "wb")
  pickle.dump(word_idx, output)
  output.close()
  
saved= open("./r8.pkl", "rb")
word_idx_saved= pickle.load(saved)
saved.close()
print("word_idx",len(word_idx_saved))
reverse_word_map = dict(map(reversed, word_idx_saved.items()))

def encoding_sent(text):
    feature_set = np.zeros((len(text), len(word_idx_saved)), dtype=np.uint32)
    tnum=0
    for t in text:
        for w in t:
            if (w in word_idx_saved):
                idx = word_idx_saved[w]
                feature_set[tnum][idx-1] = 1
        tnum += 1
    return feature_set

X_train_total= encoding_sent(train_total_list)
X_test_total= encoding_sent(test_total_list)

X_train, X_test, Y_train, Y_test = train_test_split(X_train_total, train_label, random_state=42,shuffle=True, test_size=0.10)

print(X_train.shape, Y_train.shape, X_test.shape, Y_test.shape)
CLASSES=list(set((Y_test))) #list of classes
print(CLASSES)
NUM_FEATURES=len(X_train[0])
print("NUM_FEATURES :",NUM_FEATURES)
NUM_CLAUSES=5000
print("NUM_CLAUSES :",NUM_CLAUSES)
threshold=[ 50, 100,150]       # [200, 200, 200, 200] 100*100
print("THRESHOLD :",threshold)
S= [  5.0, 10.0]      #[25.0] 10
print("S :",S)

# best, window=5, 100*100, 10.0
print("Training started")
#passing parameters and data to tsetlin machine
import csv
max_acc=0.0
for S in S:
    for THRESHOLD in threshold:
        print("S", S , "Threshold", THRESHOLD )
        #tm = MultiClassTsetlinMachine(NUM_CLAUSES, THRESHOLD, S, indexed=False, append_negated=True, weighted_clauses=False)
        #tm = MultiClassTsetlinMachine(NUM_CLAUSES, THRESHOLD, S, append_negated=True)
        tm =  MultiClassTsetlinMachine(NUM_CLAUSES, THRESHOLD, S, append_negated=True)
        print("\nAccuracy over 100 epochs:\n")
        time_diff=[]
        acc_train=[]
        acc_test=[]
        f1_score_test=[]
        for i in range(100):
            #output_file = open('false_final5_weighted.txt','a')
            tm.fit(X_train_total, train_label, epochs=1, incremental=True) 
            result = 100*(tm.predict(X_train_total) == train_label).mean()
            print("#%d r8_bits Train Accuracy: %.2f%% " % (i+1, result))
            result_valid = 100*(tm.predict(X_test) == Y_test).mean()
            print("#%d r8_bits validation Accuracy: %.2f%% " % (i+1, result_valid))
            result_test = 100*(tm.predict(X_test_total) == test_label).mean()
            print("#%d r8_bits test Accuracy: %.2f%% " % (i+1, result_test))
            f1= f1_score(test_label, tm.predict(X_test_total), average='macro' )
            f1_score_test.append(round(f1,4))
            acc_train.append(round(result,4))
            acc_test.append(round(result_test,4))
            score_known= tm.score(X_train)
            score_novel= tm.score(X_test_total)
            if result > max_acc:
                max_acc= result
                print("max", max_acc)
                ta_state= tm.get_state()
            '''
            if result_test > max_acc:
                max_acc= result_test
                print("max", max_acc)
                ta_state= tm.get_state()            
            
            '''

        with open('r8.csv', 'a+',newline='') as file:
            writer=csv.writer(file)
            writer.writerows(zip(acc_train, acc_test))
'''
hp=np.array([NUM_CLAUSES, THRESHOLD, S, NUM_FEATURES,CLASSES, len(Y_train)], dtype=object)
np.savez_compressed('R8.npz', ta_state= ta_state, hyperparameters= hp)

ld=np.load('./R8.npz', allow_pickle=True)
hyper= ld['hyperparameters']
state=ld['ta_state']
tm = MultiClassTsetlinMachine(int(hyper[0]), int(hyper[1]), int(hyper[2]), append_negated=True)
X_dummy= np.ones((int(hyper[5]), int(hyper[3])))
Y_dummy= np.random.randint(int(len(hyper[4])), size= (int(hyper[5]),))
tm.fit(X_dummy, Y_dummy, epochs=0)
tm.set_state(state)
result_saved = 100*(tm.predict(X_test_total) == test_label).mean()
print(" Accuracy_saved: " , (result_saved))

'''               

tm.set_state(ta_state)

print("\nTransforming datasets")

X_train_transformed = tm.transform(X_train_total)
X_test_transformed = tm.transform(X_test_total)

print("Saving transformed datasets")
np.savez_compressed("./X_train_transformed_r8_early_negoff.npz", X_train_transformed)
np.savez_compressed("./X_test_transformed_r8_early_negoff.npz", X_test_transformed)

In [1]:
import numpy as np
import os
os.environ["CUDA_VISIBLE_DEVICES"]="3"
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
import string
punct = string.punctuation
import os
from os import path
from tools import Tools
import random
# from PyTsetlinMachineCUDA.tm import MultiClassTsetlinMachine
def preprocess_text(text):
    return text
#prepare the sentneces 
path_train= 'train.txt'
doc_name_list_train, doc_content_list_train = Tools.split_labels_and_sentences(path_train)
print("Train size: %d" % len(doc_name_list_train))

path_test='test.txt'
doc_name_list_test, doc_content_list_test = Tools.split_labels_and_sentences(path_test)
print("Test size: %d" % len(doc_name_list_test))

#encode labels
label_encoder = preprocessing.LabelEncoder()
label = label_encoder.fit_transform(doc_name_list_train+doc_name_list_test)
label_train = label[:len(doc_name_list_train)]
label_test= label[len(doc_name_list_train):]

print(len(doc_content_list_train), len(label_train), len(doc_content_list_test), len(label_test))

#tocknize the lists as words
train_total_list= [Tools.tokenize(t) for t in doc_content_list_train]
train_label= np.array(label_train)

test_total_list= [Tools.tokenize(t) for t in doc_content_list_test]
test_label= np.array(label_test)

#involve knowledge
vectorizer_X = Tools.read_pickle_data("vectorizer_X.pickle")
for train_item, label in zip(train_total_list, doc_name_list_train):
    related_knowledge = Tools.get_related(label,vectorizer_X=vectorizer_X,top_weight=2)
    if related_knowledge is not None:
        train_item.extend(random.sample(related_knowledge, k=min(len(related_knowledge), 2)))

#prepare for encoding
print("test_label", test_label)
np_total=  train_total_list + test_total_list
words, counts = np.unique(Tools.flatten(np_total), return_counts=True)
freq, wrds = (list(i) for i in zip(*(sorted(zip(counts, words), reverse=True))))

n = 10000
features = wrds[0:n]
word_idx_saved, reverse_word_map = Tools.generate_words_map(wrds)
X_train_total= Tools.encoding_sent(train_total_list, word_idx_saved)
X_test_total= Tools.encoding_sent(test_total_list, word_idx_saved)

X_train, X_test, Y_train, Y_test = train_test_split(X_train_total, train_label, random_state=42,shuffle=True, test_size=0.10)

print(X_train.shape, Y_train.shape, X_test.shape, Y_test.shape)
CLASSES=list(set((Y_test))) #list of classes
print(CLASSES)
NUM_FEATURES=len(X_train[0])
print("NUM_FEATURES :",NUM_FEATURES)
NUM_CLAUSES=5000
print("NUM_CLAUSES :",NUM_CLAUSES)
threshold=[ 50, 100,150]       # [200, 200, 200, 200] 100*100
print("THRESHOLD :",threshold)
S= [  5.0, 10.0]      #[25.0] 10
print("S :",S)

# best, window=5, 100*100, 10.0
print("Training started")
#passing parameters and data to tsetlin machine
import csv
max_acc=0.0
for S in S:
    for THRESHOLD in threshold:
        print("S", S , "Threshold", THRESHOLD )
        tm =  MultiClassTsetlinMachine(NUM_CLAUSES, THRESHOLD, S, append_negated=True)
        print("\nAccuracy over 100 epochs:\n")
        time_diff=[]
        acc_train=[]
        acc_test=[]
        f1_score_test=[]
        for i in range(100):
            output_file = open('false_final5_weighted.txt','a')
            tm.fit(X_train_total, train_label, epochs=1, incremental=True) 
            result = 100*(tm.predict(X_train_total) == train_label).mean()
            print("#%d r8_bits Train Accuracy: %.2f%% " % (i+1, result))
            result_valid = 100*(tm.predict(X_test) == Y_test).mean()
            print("#%d r8_bits validation Accuracy: %.2f%% " % (i+1, result_valid))
            result_test = 100*(tm.predict(X_test_total) == test_label).mean()
            print("#%d r8_bits test Accuracy: %.2f%% " % (i+1, result_test))
            f1= f1_score(test_label, tm.predict(X_test_total), average='macro' )
            f1_score_test.append(round(f1,4))
            acc_train.append(round(result,4))
            acc_test.append(round(result_test,4))
            score_known= tm.score(X_train)
            score_novel= tm.score(X_test_total)
            if result > max_acc:
                max_acc= result
                print("max", max_acc)
                ta_state= tm.__getstate__()


        with open('r8.csv', 'a+',newline='') as file:
            writer=csv.writer(file)
            writer.writerows(zip(acc_train, acc_test))
          
tm.__setstate__(ta_state)

print("\nTransforming datasets")

X_train_transformed = tm.transform(X_train_total)
X_test_transformed = tm.transform(X_test_total)

print("Saving transformed datasets")
np.savez_compressed("./X_train_transformed_r8_early_negoff.npz", X_train_transformed)
np.savez_compressed("./X_test_transformed_r8_early_negoff.npz", X_test_transformed)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Ahmed\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Ahmed\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Train size: 5485
Test size: 2189
5485 5485 2189 2189


KeyboardInterrupt: 